In [ ]:
# SkinNova — Gemma 4 E2B LoRA (plan §8, test.md §5 L1–L4). Generated by ml/llm/notebooks/make_kaggle.py — edit there.
MODE = "smoke"
import os
# Kaggle's T4 accelerator is 2×T4; Unsloth's auto device map split the model across cuda:0/cuda:1 and generate() failed
# ("Expected all tensors to be on the same device"). One T4 holds 16-bit E2B + LoRA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install -q "huggingface_hub>=1.5.0,<2.0" torchcodec
!pip install -q --no-deps --upgrade timm


In [ ]:
import glob, json, os, random, sys, time, re, math
import torch
torch._dynamo.config.recompile_limit = 64
DATA = os.path.dirname(glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)[0])
sys.path.insert(0, DATA + "/code")
from ml.llm.validate import validate, guard_text, count_sentences, load_rx_terms   # same validator as the app (OutputParser.kt)
from ml.voice.intake import validate_intake
OUT = "/kaggle/working"; os.makedirs(f"{OUT}/reports", exist_ok=True)
LABELS = json.load(open(f"{DATA}/code/labels.json")); KEYS = [c["key"] for c in LABELS["classes"]]
FLOOR = {c["key"]: c["tier_floor"] for c in LABELS["classes"]}
RX = [l.strip().lower() for l in open(f"{DATA}/code/rx_terms.txt") if l.strip() and not l.startswith("#")]
TR = {t: i for i, t in enumerate(["LOW", "MODERATE", "HIGH", "URGENT"])}
def load(n): return [json.loads(l) for l in open(f"{DATA}/{n}.jsonl")]
train, sval, lval = load("sft_train"), load("sft_val"), load("llm_val")
general = load("general_regression")
print("MODE", MODE, "| train", len(train), "sft_val", len(sval), "llm_val", len(lval), "| GPU", torch.cuda.get_device_name(0))
REPORT = {"mode": MODE, "gpu": torch.cuda.get_device_name(0), "dataset_meta": json.load(open(f"{DATA}/sft_data.json"))}
def save_report():
    json.dump(REPORT, open(f"{OUT}/reports/{MODE}.json", "w"), indent=1, default=str)


In [ ]:
from PIL import Image
def to_conv(r, with_answer=True, image_override=None):
    msgs = []
    for m in r["messages"]:
        if m["role"] == "system":
            # one-part list, exactly as the app sends it (LiteRT-LM Kotlin: systemInstruction = Contents.of(text) → JSON list);
            # the stock Gemma 4 template renders it as "<text> <turn|>". (A plain string here was turned into a list by
            # UnslothVisionDataCollator anyway, and Unsloth's own gemma-4 template then printed the Python repr of that list.)
            msgs.append({"role": "system", "content": [{"type": "text", "text": m["content"]}]}); continue
        if m["role"] == "assistant" and not with_answer:
            continue
        content = []
        for c in m["content"]:
            if c["type"] == "image":
                img = image_override if image_override is not None else Image.open(f'{DATA}/{r["image"]}').convert("RGB")
                content.append({"type": "image", "image": img})
            else:
                content.append({"type": "text", "text": c["text"]})
        msgs.append({"role": m["role"], "content": content})
    return {"messages": msgs}
def target(r): return r["messages"][-1]["content"][0]["text"]


In [ ]:
from unsloth import FastVisionModel
t0 = time.time()
try:
    model, processor = FastVisionModel.from_pretrained("unsloth/gemma-4-E2B-it", load_in_4bit=False, use_gradient_checkpointing="unsloth")
    REPORT["load_in_4bit"] = False
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    model, processor = FastVisionModel.from_pretrained("unsloth/gemma-4-E2B-it", load_in_4bit=True, use_gradient_checkpointing="unsloth")
    REPORT["load_in_4bit"] = True
# Keep the checkpoint's own chat template (chat_template.jinja): it is the template `litert-torch --use_jinja_template`
# bundles into the .litertlm, so training renders prompts exactly like the phone. Unsloth's get_chat_template("gemma-4")
# printed a list-valued system turn as a Python repr (smoke v4: +13 tokens of "[{'type': 'text', ...").
assert "<|turn>system" in (processor.chat_template or ""), "expected the stock Gemma 4 chat template"
model = FastVisionModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    random_state=3407, use_rslora=False, loftq_config=None, target_modules="all-linear")
REPORT["load_s"] = round(time.time() - t0, 1)
REPORT["trainable_params"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(REPORT)


In [ ]:
# ---- L1 pre-training checks (must print PASS) ----
from unsloth.trainer import UnslothVisionDataCollator
collator = UnslothVisionDataCollator(model, processor, train_on_responses_only=True,
                                     instruction_part="<|turn>user\n", response_part="<|turn>model\n", max_seq_length=2048)
s = to_conv(next(r for r in train if r["task"] == "T1"))
txt = processor.apply_chat_template(s["messages"], tokenize=False, add_generation_prompt=False)
assert txt.count("<bos>") <= 1 and "<|turn>model" in txt and "<|turn>system" in txt, txt[:400]
assert txt.index("<|image|>") < txt.index("IMAGE_MODEL_TOP3"), "image must come before text"
# template parity: training text == generation prompt + answer + end-of-turn, system turn verbatim (app form), no repr
def _tmpl(conv):
    return [{"role": m["role"], "content": [{"type": "image"} if c["type"] == "image" else {"type": "text", "text": c["text"]} for c in m["content"]]}
            for m in conv]
n_par = 0
for r in random.Random(1).sample(train, 60):
    full = processor.apply_chat_template(_tmpl(to_conv(r)["messages"]), tokenize=False, add_generation_prompt=False)
    prompt = processor.apply_chat_template(_tmpl(to_conv(r, with_answer=False)["messages"]), tokenize=False, add_generation_prompt=True)
    assert full == prompt + target(r) + "<turn|>\n", (r["id"], full[len(prompt) - 40: len(prompt) + 80])
    assert "<|turn>system\n" + r["messages"][0]["content"].strip() + " <turn|>\n" in full and "[{'" not in full, r["id"]
    n_par += 1
batch = collator([s]); lab = batch["labels"][0]; ids = batch["input_ids"][0]
seen = processor.tokenizer.decode(ids)
assert s["messages"][0]["content"][0]["text"].strip() in seen and "[{'" not in seen, "collator changed the system turn"
visible = processor.tokenizer.decode(ids[lab != -100])
assert visible.strip().startswith("{") and "IMAGE_MODEL_TOP3" not in visible, visible[:300]
lens = []
for r in random.Random(0).sample(train, 200):
    b = collator([to_conv(r)]); lens.append(int(b["input_ids"].shape[1]))
lens.sort(); p99 = lens[int(.99 * len(lens)) - 1]
REPORT["L1"] = {"template_ok": True, "template_parity_checked": n_par, "masking_ok": True, "visible_head": visible[:120], "tokens_p50": lens[100], "tokens_p99": p99, "tokens_max": lens[-1]}
assert p99 <= 2048, p99
print("L1 PASS", REPORT["L1"]); save_report()


In [ ]:
# ---- inference helpers ----
def gen(r, max_new_tokens=700, image_override=None, adapter=True):
    conv = to_conv(r, with_answer=False, image_override=image_override)["messages"]
    imgs = [c["image"] for m in conv if isinstance(m["content"], list) for c in m["content"] if c["type"] == "image"]
    tmpl = [{"role": m["role"], "content": m["content"] if isinstance(m["content"], str) else
             [{"type": "image"} if c["type"] == "image" else {"type": "text", "text": c["text"]} for c in m["content"]]} for m in conv]
    text = processor.apply_chat_template(tmpl, add_generation_prompt=True)
    inputs = (processor(imgs[0], text, add_special_tokens=False, return_tensors="pt") if imgs else
              processor(text=text, add_special_tokens=False, return_tensors="pt")).to("cuda")
    def _g():
        return model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    if adapter:
        out = _g()
    else:
        with model.disable_adapter():
            out = _g()
    return processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def gen_many(recs, max_new_tokens=700, image_override=None, adapter=True, bs=8):
    """Batched greedy generation (left padding); falls back to one-by-one if the processor rejects a batch."""
    outs = []
    tok = processor.tokenizer; tok.padding_side = "left"
    for s0 in range(0, len(recs), bs):
        chunk = recs[s0:s0 + bs]
        convs = [to_conv(r, with_answer=False, image_override=image_override)["messages"] for r in chunk]
        imgs = [[c["image"] for m in cv if isinstance(m["content"], list) for c in m["content"] if c["type"] == "image"] for cv in convs]
        texts = [processor.apply_chat_template([{"role": m["role"], "content": m["content"] if isinstance(m["content"], str) else
                  [{"type": "image"} if c["type"] == "image" else {"type": "text", "text": c["text"]} for c in m["content"]]} for m in cv],
                  add_generation_prompt=True) for cv in convs]
        try:
            if all(imgs) and len(set(len(i) for i in imgs)) == 1:
                inputs = processor(images=imgs, text=texts, add_special_tokens=False, padding=True, return_tensors="pt").to("cuda")
            elif not any(imgs):
                inputs = processor(text=texts, add_special_tokens=False, padding=True, return_tensors="pt").to("cuda")
            else:
                raise ValueError("mixed image/text batch")
            def _g():
                return model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
            if adapter:
                out = _g()
            else:
                with model.disable_adapter():
                    out = _g()
            n = inputs["input_ids"].shape[1]
            outs += [tok.decode(o[n:], skip_special_tokens=True) for o in out]
        except Exception as e:
            print("batch fallback:", type(e).__name__, str(e)[:120])
            outs += [gen(r, max_new_tokens, image_override, adapter) for r in chunk]
    return outs

from ml.eval.llm_metrics import score_analysis as _sa, score_extract as _se, summarize, summarize_extract, ANALYSIS_KEYS
def score_analysis(r, text): return _sa(r["meta"], text)
def score_extract(r, text): return _se(r["meta"], text)


In [ ]:
# ---- base-model sanity (E0) on a few llm_val cases: proves the eval path works before any training ----
FastVisionModel.for_inference(model)
t0 = time.time(); cases = [x for x in lval if x["task"] == "T1"][:4]
outs = gen_many(cases, 700, adapter=False, bs=4)
base = [{"id": r["id"], "out": o[:1500], **score_analysis(r, o)} for r, o in zip(cases, outs)]
REPORT["E0_sanity"] = {"cases": base, "s_per_case_batched": round((time.time() - t0) / len(cases), 1)}
print(json.dumps(REPORT["E0_sanity"], indent=1)[:3000]); save_report()


In [ ]:
from trl import SFTTrainer, SFTConfig
def sft_args(**kw):
    base = dict(per_device_train_batch_size=1, gradient_accumulation_steps=8, max_grad_norm=0.3, warmup_ratio=0.03,
                learning_rate=2e-4, logging_steps=5, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="cosine",
                seed=3407, output_dir=f"{OUT}/out", report_to="none", remove_unused_columns=False, dataset_text_field="",
                dataset_kwargs={"skip_prepare_dataset": True}, max_length=2048, save_strategy="no")
    base.update(kw); return SFTConfig(**base)

if MODE == "smoke":
    # ---- L2 overfit test: 16 records, 100 steps → loss < 0.3 and ≥ 14/16 exact greedy reproductions ----
    # v4: smoke v3 reached loss 0.021 but only 3/16 exact; every divergence sat on a phrase the generator picks at random.
    # So: (a) teacher-forced token check through BOTH the training (collator) path and the inference path — a template or
    # image-token mismatch shows up as a much higher inference-path NLL; (b) batched vs one-by-one generation must agree;
    # (c) keep training in rounds of 100 steps and re-measure, to see whether exact reproduction is a capacity/steps issue.
    os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
    small = [r for r in train if r["task"] in {"T1", "T2", "T3", "T4", "T9"}][:12] + [r for r in train if r["task"] in {"T6", "T7", "T8"}][:4]
    RESP_IDS = processor.tokenizer.encode("<|turn>model\n", add_special_tokens=False)
    def _prompt_inputs(r, suffix=""):
        conv = to_conv(r, with_answer=False)["messages"]
        imgs = [c["image"] for m in conv if isinstance(m["content"], list) for c in m["content"] if c["type"] == "image"]
        tmpl = [{"role": m["role"], "content": m["content"] if isinstance(m["content"], str) else
                 [{"type": "image"} if c["type"] == "image" else {"type": "text", "text": c["text"]} for c in m["content"]]} for m in conv]
        text = processor.apply_chat_template(tmpl, add_generation_prompt=True) + suffix
        return (processor(imgs[0], text, add_special_tokens=False, return_tensors="pt") if imgs else
                processor(text=text, add_special_tokens=False, return_tensors="pt")).to("cuda")
    @torch.no_grad()
    def teacher_forced(r, path):
        if path == "collator":
            b = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in collator([to_conv(r)]).items()}
            lab = b.pop("labels")[0]; ids = b["input_ids"][0]; pos = (lab != -100).nonzero().flatten()
        else:
            b = _prompt_inputs(r, target(r) + "<turn|>\n"); ids = b["input_ids"][0]; L = ids.tolist(); n = len(RESP_IDS)
            st = max(i for i in range(len(L) - n + 1) if L[i:i + n] == RESP_IDS) + n
            pos = torch.arange(st, len(L), device=ids.device)
        logits = model(**b).logits[0]
        lp = torch.log_softmax(logits[pos - 1].float(), -1); tgt = ids[pos]
        pt = lp.gather(1, tgt[:, None]).squeeze(1); am = lp.argmax(-1)
        mism = (am != tgt).nonzero().flatten().tolist(); dec = processor.tokenizer.decode
        return {"n_input_tokens": int(ids.shape[0]), "pixel_values": list(b["pixel_values"].shape) if "pixel_values" in b else None,
                "n_target": int(tgt.shape[0]), "nll_mean": round(float(-pt.mean()), 4), "nll_sum": round(float(-pt.sum()), 2),
                "argmax_mismatches": len(mism),
                "first_mismatches": [{"context": dec(tgt[max(0, i - 10):i]), "target": dec(tgt[i:i + 1]), "argmax": dec(am[i:i + 1]),
                                      "p_target": round(float(pt[i].exp()), 3)} for i in mism[:5]]}
    def decisions(o, task):
        if task == "T7":
            return o.strip()            # narration: plain text
        try:
            j = json.loads(o)
        except Exception:
            return None
        if isinstance(j, dict) and "possible_categories" in j:
            try:
                return ([(c["key"], c["likelihood"]) for c in j["possible_categories"]], j["uncertainty"]["level"],
                        j["triage"]["tier"], j.get("disagreement_with_image_model"))
            except Exception:
                return None
        return j                         # extraction / translation: the whole object
    def evaluate_small(tag):
        FastVisionModel.for_inference(model)
        exact = jsoneq = deceq = 0; outs = []
        for r, o in zip(small, gen_many(small, 700, bs=4)):
            o = o.strip(); t = target(r).strip(); ok = o == t; exact += ok
            try:
                je = json.loads(o) == json.loads(t)
            except Exception:
                je = False
            jsoneq += je
            dec = decisions(o, r["task"]); dec_ok = dec is not None and dec == decisions(t, r["task"]); deceq += dec_ok
            diff = next((i for i, (a, b) in enumerate(zip(o, t)) if a != b), min(len(o), len(t)))
            outs.append({"id": r["id"], "task": r["task"], "exact": ok, "json_equal": je, "decisions_equal": dec_ok, "first_diff": diff, "len_out": len(o), "len_target": len(t),
                         "out_at_diff": o[max(0, diff - 60): diff + 60], "target_at_diff": t[max(0, diff - 60): diff + 60], "out": o})
        print(tag, "exact", exact, "json_equal", jsoneq, "decisions_equal", deceq, flush=True)
        return exact, jsoneq, deceq, outs
    rounds = []; total_steps = 0; t_train = 0.0
    for rnd in range(2):
        FastVisionModel.for_training(model)
        trainer = SFTTrainer(model=model, train_dataset=[to_conv(r) for r in small], processing_class=processor.tokenizer,
                             data_collator=collator, args=sft_args(gradient_accumulation_steps=1, max_steps=100, warmup_ratio=0.0))
        t0 = time.time(); trainer.train(); dt = time.time() - t0; t_train += dt; total_steps += 100
        losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
        exact, jsoneq, deceq, outs = evaluate_small(f"after {total_steps} steps")
        rec = {"steps": total_steps, "losses": losses, "final_loss": losses[-1], "exact": exact, "json_equal": jsoneq, "decisions_equal": deceq,
               "train_s": round(dt, 1), "outs": outs}
        if rnd == 0:
            REPORT["L2"] = {"first_loss": losses[0], "final_loss": losses[-1], "losses": losses, "exact": exact, "json_equal": jsoneq, "n": len(small),
                            "decisions_equal": deceq, "train_s": round(dt, 1), "s_per_step_bs1": round(dt / 100, 2),
                            "peak_mem_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2), "outs": outs}
            steps_full = math.ceil(3040 / 8)   # full plan: ~3,040 records × 1 epoch
            REPORT["projection"] = {"full_steps": steps_full, "est_full_train_h": round(steps_full * 8 * dt / 100 / 3600, 2)}
            try:   # (a) training path vs inference path, same weights: same token count and ~same loss expected
                REPORT["L2_teacher_forced"] = [{"id": r["id"], "task": r["task"], "collator": teacher_forced(r, "collator"),
                                                "inference": teacher_forced(r, "inference")} for r in small]
                REPORT["L2_path_parity"] = sum(x["collator"]["n_input_tokens"] == x["inference"]["n_input_tokens"] for x in REPORT["L2_teacher_forced"])
            except Exception as e:
                REPORT["L2_teacher_forced"] = f"error: {type(e).__name__}: {e}"
            try:   # (b) batched (left-padded) vs one-by-one greedy outputs on the first 4 image records
                single = [gen(r, 700).strip() for r in small[:4]]
                REPORT["L2_batch_consistency"] = {"same": sum(a == o["out"] for a, o in zip(single, outs[:4])), "n": 4,
                                                  "single": [s_[:300] for s_ in single]}
            except Exception as e:
                REPORT["L2_batch_consistency"] = f"error: {type(e).__name__}: {e}"
        rounds.append(rec); REPORT["L2_rounds"] = rounds
        # test.md L2 (rev. 2026-10-06): loss < 0.3 and the record-specific decisions (categories + order, likelihoods,
        # uncertainty level, tier, disagreement; whole object for extraction/translation; text for narration) reproduce
        # for ≥ 14/16 within 200 steps. Word-for-word output is reported, not gated: the generator draws paraphrases at random.
        REPORT["L2"]["pass"] = bool(losses[-1] < 0.3 and deceq >= 14)
        REPORT["L2"]["pass_at_steps"] = REPORT["L2"].get("pass_at_steps") or (total_steps if REPORT["L2"]["pass"] else None)
        save_report()
    print("L2", "PASS" if REPORT["L2"]["pass"] else "FAIL", {k: REPORT["L2"][k] for k in ["first_loss", "final_loss", "exact", "train_s", "peak_mem_gb"]},
          REPORT["projection"], "| rounds:", [(x["steps"], round(x["final_loss"], 4), x["exact"], x["json_equal"], x["decisions_equal"]) for x in rounds],
          "| path parity", REPORT.get("L2_path_parity"))
    if isinstance(REPORT.get("L2_teacher_forced"), list):
        for x in REPORT["L2_teacher_forced"]:
            print(x["id"], "collator nll", x["collator"]["nll_mean"], "tok", x["collator"]["n_input_tokens"], "| inference nll", x["inference"]["nll_mean"],
                  "tok", x["inference"]["n_input_tokens"], "| mism", x["collator"]["argmax_mismatches"], x["inference"]["argmax_mismatches"])
    print("batch consistency", REPORT.get("L2_batch_consistency", {}).get("same") if isinstance(REPORT.get("L2_batch_consistency"), dict) else REPORT.get("L2_batch_consistency"))
    save_report()


In [ ]:
if MODE == "full":
    # ---- E1: full training (plan §8.3), eval loss on sft_val ----
    FastVisionModel.for_training(model)
    trainer = SFTTrainer(model=model, train_dataset=[to_conv(r) for r in train], eval_dataset=[to_conv(r) for r in sval],
                         processing_class=processor.tokenizer, data_collator=collator,
                         args=sft_args(num_train_epochs=1, eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
                                       save_strategy="steps", save_steps=100, save_total_limit=2, load_best_model_at_end=True,
                                       metric_for_best_model="eval_loss", greater_is_better=False))
    t0 = time.time(); trainer.train(); dt = time.time() - t0
    REPORT["train"] = {"wall_h": round(dt / 3600, 2), "log": trainer.state.log_history, "best": trainer.state.best_metric,
                       "peak_mem_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2)}
    model.save_pretrained(f"{OUT}/lora"); processor.save_pretrained(f"{OUT}/lora")
    save_report()


In [ ]:
if MODE == "full":
    # ---- L4: E0 (adapter off) vs E1 on llm_val; image ablation; extraction; forgetting set ----
    FastVisionModel.for_inference(model)
    an = [r for r in lval if r["task"] in {"T1", "T9"}][:40]
    ex = [r for r in lval if r["task"] == "T6"][:20]
    res = {}
    for arm, adapter in [("E1", True), ("E0", False)]:
        outs = gen_many(an, 700, adapter=adapter)
        res[arm] = [{"id": r["id"], "task": r["task"], **score_analysis(r, o), "out": o[:1500]} for r, o in zip(an, outs)]
        save_report()
    REPORT["L4_analysis"] = {arm: summarize(rows, ANALYSIS_KEYS) for arm, rows in res.items()}
    REPORT["L4_analysis_rows"] = res
    gray = Image.new("RGB", (512, 512), (128, 128, 128))
    gouts = gen_many(an[:16], 700, image_override=gray)
    abl = [{"id": r["id"], "real": row["cats"], "gray": score_analysis(r, o)["cats"]} for r, o, row in zip(an[:16], gouts, res["E1"][:16])]
    for x in abl: x["changed"] = set(x["real"]) != set(x["gray"])
    REPORT["image_ablation"] = {"changed_rate": round(sum(x["changed"] for x in abl) / len(abl), 3), "rows": abl}
    xr = {}
    for arm, adapter in [("E1", True), ("E0", False)]:
        outs = gen_many(ex, 300, adapter=adapter)
        xr[arm] = [{"id": r["id"], **score_extract(r, o), "out": o[:600]} for r, o in zip(ex, outs)]
    REPORT["L4_extract"] = {arm: summarize_extract(rows) for arm, rows in xr.items()}
    gq = [{"messages": [{"role": "user", "content": [{"type": "text", "text": q["prompt"]}]}]} for q in general[:20]]
    g1, g0 = gen_many(gq, 150), gen_many(gq, 150, adapter=False)
    REPORT["general_regression"] = [{"prompt": q["prompt"], "E1": a1, "E0": a0} for q, a1, a0 in zip(general, g1, g0)]
    save_report()
    print(json.dumps({k: REPORT[k] for k in ["L4_analysis", "L4_extract"]}, indent=1), REPORT["image_ablation"]["changed_rate"])


In [ ]:
if MODE == "full":
    # merged 16-bit weights for the .litertlm export notebook (test L5 merge parity runs there)
    model.save_pretrained_merged(f"{OUT}/merged", processor, save_method="merged_16bit")
    print(os.listdir(f"{OUT}/merged"))
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN"); user = UserSecretsClient().get_secret("HF_USER")
        model.push_to_hub(f"{user}/skinnova-e2b-lora-v1", token=tok, private=True); processor.push_to_hub(f"{user}/skinnova-e2b-lora-v1", token=tok, private=True)
        REPORT["hf_lora_repo"] = f"{user}/skinnova-e2b-lora-v1"
    except Exception as e:
        REPORT["hf_push"] = f"skipped: {type(e).__name__}"
    save_report()
